In [1]:
!pip install vllm transformers accelerate

In [2]:
!pip install litellm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 14.0 MB/s eta 0:00:00


In [3]:
!nohup vllm serve Qwen/Qwen3-1.7B \
  --enable-auto-tool-choice \
  --tool-call-parser hermes \
  > vllm.log &

nohup: redirecting stderr to stdout


In [16]:
!tail -n 20 vllm.log # to watch last 20 rows og vllm.log, about 2/3 minutes to have the server ready -> Application startup complete

(APIServer pid=2038) INFO 11-17 08:22:09 [launcher.py:42] Route: /v1/responses/{response_id}/cancel, Methods: POST
(APIServer pid=2038) INFO 11-17 08:22:09 [launcher.py:42] Route: /v1/chat/completions, Methods: POST
(APIServer pid=2038) INFO 11-17 08:22:09 [launcher.py:42] Route: /v1/completions, Methods: POST
(APIServer pid=2038) INFO 11-17 08:22:09 [launcher.py:42] Route: /v1/embeddings, Methods: POST
(APIServer pid=2038) INFO 11-17 08:22:09 [launcher.py:42] Route: /pooling, Methods: POST
(APIServer pid=2038) INFO 11-17 08:22:09 [launcher.py:42] Route: /classify, Methods: POST
(APIServer pid=2038) INFO 11-17 08:22:09 [launcher.py:42] Route: /score, Methods: POST
(APIServer pid=2038) INFO 11-17 08:22:09 [launcher.py:42] Route: /v1/score, Methods: POST
(APIServer pid=2038) INFO 11-17 08:22:09 [launcher.py:42] Route: /v1/audio/transcriptions, Methods: POST
(APIServer pid=2038) INFO 11-17 08:22:09 [launcher.py:42] Route: /v1/audio/translations, Methods: POST
(APIServer pid=2038) INFO 11-

In [25]:
import sqlite3
db_name = "shakespeare.sqlite"
conn = sqlite3.connect(db_name)

In [26]:
import pandas as pd

def query_db(sql: str):
    """
    Executes an SQL query and returns the result as JSON

    Args:
        sql: The SQL query to execute

    Returns:
        The result of the query as JSON
    """
    try:
        df = pd.read_sql_query(sql, conn)
        return df.to_dict(orient="records")
    except Exception as e:
        return {"error": str(e)}

In [27]:
def get_schema(connection):
    schema_str = ""
    cursor = connection.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    for table_name in tables:
        table_name = table_name[0]
        if table_name.startswith("sqlite_"):
            continue

        cursor.execute(f"SELECT sql FROM sqlite_master WHERE type='table' AND name='{table_name}';")
        create_statement = cursor.fetchone()[0]
        schema_str += create_statement + ";\n"

    return schema_str

In [28]:
db_schema = get_schema(conn)

In [29]:
def get_function_by_name(name):
    if name == "query_db":
        return query_db

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "query_db",
            "description": "Executes a single SQL query on the database. Use this to make exploratory queries in order to find the final answer.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {
                        "type": "string",
                        "description": 'The SQL query to execute on the database.',
                    }
                },
                "required": ["sql"],
            },
        },
    }
]

In [30]:
USER_QUESTION = f"""
Database Schema:
{db_schema}
Evidence:
Twelfth Night refers to Title = 'Twelfth Night'
Question:
How many scenes are there in Act 1 in Twelfth Night?.
"""



SYSTEM_PROMPT = """You are a SQL expert. Your task is to answer the user’s question by generating a final SQL query.
The schema of the database is provided in the user question.
You can use the evidence provided in the user question to ensure that ambiguous natural language terms are correctly translated into the specific SQL logic required by the database schema.
You have access to a tool called `query_db` that you can use to run exploratory queries on the database. Use it only if needed to check values and validate assumptions.
When you perform exploratory queries, use LIMIT 3 if the output of the query may have more than 3 rows.
"""

In [31]:
from litellm import completion
import json

pred_query = None
model_name = "hosted_vllm/Qwen/Qwen3-1.7B"
MAX_TURNS = 10 # a maximum number of turns in order to avoid infinite loops

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_QUESTION}
]

completion_params = {
    "model": model_name,
    "temperature": 0.1,
    "top_p": 0.8,
    "max_tokens": 4096,
    "repetition_penalty": 1.05,
    "api_base": "http://127.0.0.1:8000/v1",
    "tools": TOOLS
}

print(USER_QUESTION)

for i in range(MAX_TURNS):

    # call the model
    response = completion(messages=messages, **completion_params)
    response_message = response.choices[0].message

    # add the model's response (thought + tool_call) to the history
    messages.append(response_message.model_dump())

    # check if the model has called a tool
    tool_calls = response_message.get("tool_calls", None)
    if tool_calls:
        # execution of the tool call
        for tool_call in tool_calls:
            call_id = tool_call["id"]
            if fn_call := tool_call.get("function"):
                fn_name = fn_call["name"]
                fn_args = json.loads(fn_call["arguments"])
                sql_query = fn_args.get("sql", "")
                pred_query = sql_query.strip()
                fn_res = get_function_by_name(fn_name)(**fn_args)

                #add the tool result (as JSON) to the history for the model
                messages.append({
                    "role": "tool",
                    "content": json.dumps({
                        "query_executed": sql_query,
                        "result": fn_res
                    }),
                    "tool_call_id": call_id,
                })

    # Check if the model is finished (gave a final answer without tool calls)
    elif response_message.content:
        final_output = response_message.content.strip()
        if not pred_query:
          pred_query = final_output

        print(f"Final SQL query:\n{pred_query}")

        break


conn.close()


Database Schema:
CREATE TABLE "chapters"
(
    id          INTEGER
        primary key autoincrement,
    Act         INTEGER not null,
    Scene       INTEGER not null,
    Description TEXT    not null,
    work_id     INTEGER not null
        references works
);
CREATE TABLE "characters"
(
    id          INTEGER
        primary key autoincrement,
    CharName    TEXT not null,
    Abbrev      TEXT not null,
    Description TEXT not null
);
CREATE TABLE "paragraphs"
(
    id           INTEGER
        primary key autoincrement,
    ParagraphNum INTEGER           not null,
    PlainText    TEXT              not null,
    character_id INTEGER           not null
        references characters,
    chapter_id   INTEGER default 0 not null
        references chapters
);
CREATE TABLE "works"
(
    id        INTEGER
        primary key autoincrement,
    Title     TEXT    not null,
    LongTitle TEXT    not null,
    Date      INTEGER not null,
    GenreType TEXT    not null
);

Evidence:
Twe

In [32]:
messages

[{'role': 'system',
  'content': 'You are a SQL expert. Your task is to answer the user’s question by generating a final SQL query.\nThe schema of the database is provided in the user question.\nYou can use the evidence provided in the user question to ensure that ambiguous natural language terms are correctly translated into the specific SQL logic required by the database schema.\nYou have access to a tool called `query_db` that you can use to run exploratory queries on the database. Use it only if needed to check values and validate assumptions.\nWhen you perform exploratory queries, use LIMIT 3 if the output of the query may have more than 3 rows.\nYour output must ALWAYS be a SQL query, not its result.\n'},
 {'role': 'user',
  'content': '\nDatabase Schema:\nCREATE TABLE "chapters"\n(\n    id          INTEGER\n        primary key autoincrement,\n    Act         INTEGER not null,\n    Scene       INTEGER not null,\n    Description TEXT    not null,\n    work_id     INTEGER not null\

In [33]:
def compute_execution_accuracy(gt_results, predict_results):
  num_correct = 0
  num_queries = len(gt_results)
  mismatch_idx = []

  for i, result in enumerate(gt_results):
      if set(result['results']) == set(predict_results[i]['results']):
          num_correct += 1
      else:
          mismatch_idx.append(i)

  acc = (num_correct / num_queries) * 100

  return acc

In [34]:
def run_query(db_path, query):
  conn = sqlite3.connect(db_path)
  cursor = conn.cursor()
  cursor.execute(query)
  rows = cursor.fetchall()
  conn.close()

  # Flatten results and convert to list of strings
  return [row[0] for row in rows]

In [39]:
gt_query = """SELECT COUNT(*) as n_Scenes FROM works w JOIN chapters c ON c.work_id = w.id  WHERE w.Title = 'Twelfth Night' AND c.Act = 1"""

In [40]:
rows_gt = run_query(db_name, gt_query)
gt_res = [{"results": rows_gt}]

rows_pred = run_query(db_name, pred_query)
pred_res = [{"results": rows_pred}]

In [41]:
acc = compute_execution_accuracy(gt_res, pred_res)
print(f"Accuracy of the generated SQL query: {acc}")

Accuracy of the generated SQL query: 100.0
